<a href="https://colab.research.google.com/github/Deepasivakumar25/AI_Learning/blob/main/chat_bot_with_RAG2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers pypdf transformers accelerate sentencepiece faiss-cpu

In [ ]:
from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

import faiss
import numpy as np

from transformers import pipeline,AutoTokenizer

In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
file_content = PdfReader("visa_cheklist_2.pdf")
pdf_text = ""
for page in file_content.pages:
    pdf_text += page.extract_text()


In [ ]:
example_para = "Artificial Intelligence (AI) enables computers to perform tasks that normally require human intelligence. Machine Learning is a branch of AI that allows computers to learn from data. Deep Learning is a specialized type of Machine Learning that uses neural networks with multiple layers. AI is widely used in healthcare, finance, education, and self-driving cars."

In [ ]:
chunk_list = []
chunk_size = 100
for i in range(0, len(example_para),chunk_size):
    chunk_list.append(example_para[i:i+chunk_size])
print(chunk_list)

embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2') # converting chunks into numbers

chunk_embedding = embedding_model.encode(chunk_list)
dimension = chunk_embedding.shape[1] # saving the vectors into FAISS database
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embedding)



In [ ]:
chatbot = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)
tokenizer = AutoTokenizer.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)

In [ ]:
while True:
  question = input("Ask a question: ")
  if question == "exit":
    break
  question_embedding = embedding_model.encode([question])

  distance, index_number = index.search(
    question_embedding,
    k=1)

  index_number = index_number[0][0]
  best_chunk = chunk_list[index_number]

  print(distance, index_number)

  prompt = f"""<|user|>

Use ONLY the context below.

Context:
{best_chunk}

Question:
{question}

If the answer is not present, reply:
I couldn't find that information.

<|assistant|>
"""
  print("\n==============================")
  print("ORIGINAL PROMPT")
  print("==============================")
  print(prompt)

  # Split prompt into tokens
  tokens = tokenizer.tokenize(prompt)

  print("\n==============================")
  print("TOKENS")
  print("==============================")
  print(tokens)

  # Convert tokens into IDs
  token_ids = tokenizer.encode(prompt)

  print("\n==============================")
  print("TOKEN IDS")
  print("==============================")
  print(token_ids)

  print("\nTotal Tokens:", len(token_ids))

  # Convert IDs back into text
  decoded_text = tokenizer.decode(token_ids)

  print("\n==============================")
  print("DECODED TEXT")
  print("==============================")
  print(decoded_text)
  response = chatbot(
    prompt,
    max_new_tokens=120,
    do_sample=False,

)
  answer = response[0]["generated_text"].strip().split("\n")[0]

  print("\nBot:")
  print(answer)

